In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [2]:

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


Train shape: (1460, 81)
Test shape: (1459, 80)


In [3]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [5]:
train.isnull().sum() > 0

,0
Id,False
MSSubClass,False
MSZoning,False
LotFrontage,True
LotArea,False
...,...
MoSold,False
YrSold,False
SaleType,False
SaleCondition,False


In [7]:
train_size = len(train)

train = train.set_index("Id")
test = test.set_index("Id")

# Temporary target column so train and test can be combined
test["SalePrice"] = 0

data = pd.concat([train, test], axis=0)

In [8]:
categorical_cols = data.select_dtypes(include="object").columns

high_missing_cat = [
    col for col in categorical_cols
    if data[col].isna().sum() > 1100
]

data = data.drop(columns=high_missing_cat)

categorical_cols = data.select_dtypes(include="object").columns
data[categorical_cols] = data[categorical_cols].fillna("null")

data = pd.get_dummies(data, columns=categorical_cols)

print("Encoded shape:", data.shape)

Encoded shape: (2919, 284)


In [15]:
mode_columns = [
    "GarageCars", "GarageYrBlt", "BsmtFullBath", "BsmtHalfBath"
]

mean_columns = [
    "LotFrontage", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "GarageArea"
]

for col in mode_columns:
    data[col] = data[col].fillna(data[col].mode()[0])

for col in mean_columns:
    data[col] = data[col].fillna(data[col].mean())

print("Remaining missing values:", data.isna().sum().sum())

Remaining missing values: 0


In [12]:
for i in train.columns:
    if 'null' in i:
        train = train.drop(i, axis = 1)
        print(i)

In [16]:
training_data = data.iloc[:train_size].copy()
testing_data = data.iloc[train_size:].drop(columns="SalePrice").copy()

X = training_data.drop(columns="SalePrice")
y = np.log1p(training_data["SalePrice"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Test shape:", testing_data.shape)


X shape: (1460, 283)
y shape: (1460,)
Test shape: (1459, 283)


In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (1168, 283)
Validation: (292, 283)


In [26]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1
    )
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    results[name] = rmse
    print(f"{name}: {rmse:.4f}")


Linear Regression: 0.1293
Random Forest: 0.1445
Gradient Boosting: 0.1350
XGBoost: 0.1398


In [27]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = {}

for name, model in models.items():
    scores = np.sqrt(
        -cross_val_score(
            model,
            X,
            y,
            scoring="neg_mean_squared_error",
            cv=kf,
            n_jobs=-1
        )
    )

    cv_results[name] = {
        "mean_rmse": scores.mean(),
        "std_rmse": scores.std()
    }

    print(f"{name}:")
    print(f"  Fold RMSE: {scores}")
    print(f"  Mean RMSE: {scores.mean():.4f}")
    print(f"  Std RMSE:  {scores.std():.4f}")


Linear Regression:
  Fold RMSE: [0.12930568 0.12339569 0.22973243 0.15327582 0.11280459]
  Mean RMSE: 0.1497
  Std RMSE:  0.0422
Random Forest:
  Fold RMSE: [0.14568956 0.12654075 0.17715788 0.1487798  0.12189066]
  Mean RMSE: 0.1440
  Std RMSE:  0.0196
Gradient Boosting:
  Fold RMSE: [0.13501169 0.11231505 0.17101678 0.13728655 0.11351052]
  Mean RMSE: 0.1338
  Std RMSE:  0.0213
XGBoost:
  Fold RMSE: [0.13977824 0.1313578  0.17318333 0.136234   0.11775087]
  Mean RMSE: 0.1397
  Std RMSE:  0.0184


In [28]:
gb_model = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=3,
    min_samples_leaf=3,
    min_samples_split=10,
    loss="huber",
    random_state=42
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = np.sqrt(
    -cross_val_score(
        gb_model,
        X,
        y,
        scoring="neg_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )
)

print("Fold RMSE:", scores)
print("Mean RMSE:", scores.mean())
print("Std RMSE:", scores.std())

Fold RMSE: [0.12963966 0.1112253  0.16653852 0.12819464 0.10535375]
Mean RMSE: 0.1281903729656539
Std RMSE: 0.02136354700022381


In [29]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = np.sqrt(
    -cross_val_score(
        model,
        X,
        y,
        scoring="neg_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )
)

print("Fold RMSE:", scores)
print("Mean RMSE:", scores.mean())
print("Std RMSE:", scores.std())

Fold RMSE: [0.13977824 0.1313578  0.17318333 0.136234   0.11775087]
Mean RMSE: 0.13966084716161328
Std RMSE: 0.018354845986306233


In [30]:
# Fit the final model on the complete training data
gb_model.fit(X, y)

print("Model fitted successfully.")

Model fitted successfully.


In [31]:
pred_log = gb_model.predict(testing_data)
pred = np.expm1(pred_log)

submission = pd.DataFrame({
    "Id": testing_data.index,
    "SalePrice": pred
})

submission.to_csv("output.csv", index=False)

print(submission.head())
print("\nShape:", submission.shape)
print("\nMissing values:")
print(submission.isnull().sum())


     Id      SalePrice
0  1461  122326.459053
1  1462  154559.611474
2  1463  183339.392537
3  1464  191297.831338
4  1465  189697.907762

Shape: (1459, 2)

Missing values:
Id           0
SalePrice    0
dtype: int64


In [32]:
import joblib

joblib.dump(gb_model, "house_price_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [33]:
training_data.columns.tolist()

['MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold',
 'SalePrice',
 'MSZoning_C (all)',
 'MSZoning_FV',
 'MSZoning_RH',
 'MSZoning_RL',
 'MSZoning_RM',
 'MSZoning_null',
 'Street_Grvl',
 'Street_Pave',
 'LotShape_IR1',
 'LotShape_IR2',
 'LotShape_IR3',
 'LotShape_Reg',
 'LandContour_Bnk',
 'LandContour_HLS',
 'LandContour_Low',
 'LandContour_Lvl',
 'Utilities_AllPub',
 'Utilities_NoSeWa',
 'Utilities_null',
 'LotConfig_Corner',
 'LotConfig_CulDSac',
 'LotConfig_FR2',
 'LotConfig_FR3',
 'LotConfig_Inside',
 'Land

In [42]:
# Features used by the model
model_features = training_data.drop(columns=["SalePrice"]).columns

print("Number of model features:", len(model_features))

Number of model features: 283


In [47]:
def create_user_input(
    overall_qual,
    overall_cond,
    year_built,
    year_remod_add,
    lot_area,
    gr_liv_area,
    total_bsmt_sf,
    first_flr_sf,
    second_flr_sf,
    full_bath,
    half_bath,
    bedroom_abv_gr,
    tot_rms_abv_grd,
    fireplaces,
    garage_cars,
    garage_area,
    garage_yr_blt,
    neighborhood="CollgCr",
    ms_zoning="RL",
    kitchen_qual="TA",
    central_air="Y"
):

    # Create empty dataframe with exactly the model features
    user_input = pd.DataFrame(
        0,
        index=[0],
        columns=model_features
    )

    # -------------------------
    # Numerical features
    # -------------------------

    user_input["OverallQual"] = overall_qual
    user_input["OverallCond"] = overall_cond
    user_input["YearBuilt"] = year_built
    user_input["YearRemodAdd"] = year_remod_add
    user_input["LotArea"] = lot_area
    user_input["GrLivArea"] = gr_liv_area
    user_input["TotalBsmtSF"] = total_bsmt_sf
    user_input["1stFlrSF"] = first_flr_sf
    user_input["2ndFlrSF"] = second_flr_sf
    user_input["FullBath"] = full_bath
    user_input["HalfBath"] = half_bath
    user_input["BedroomAbvGr"] = bedroom_abv_gr
    user_input["TotRmsAbvGrd"] = tot_rms_abv_grd
    user_input["Fireplaces"] = fireplaces
    user_input["GarageCars"] = garage_cars
    user_input["GarageArea"] = garage_area
    user_input["GarageYrBlt"] = garage_yr_blt

    # -------------------------
    # Categorical features
    # -------------------------

    categorical_inputs = {
        "Neighborhood": neighborhood,
        "MSZoning": ms_zoning,
        "KitchenQual": kitchen_qual,
        "CentralAir": central_air
    }

    for feature, value in categorical_inputs.items():
        column = f"{feature}_{value}"

        if column in user_input.columns:
            user_input[column] = 1

    return user_input

In [48]:
user_input = create_user_input(
    overall_qual=7,
    overall_cond=5,
    year_built=2003,
    year_remod_add=2003,
    lot_area=8450,
    gr_liv_area=1710,
    total_bsmt_sf=856,
    first_flr_sf=856,
    second_flr_sf=854,
    full_bath=2,
    half_bath=1,
    bedroom_abv_gr=3,
    tot_rms_abv_grd=7,
    fireplaces=1,
    garage_cars=2,
    garage_area=548,
    garage_yr_blt=2003,
    neighborhood="CollgCr",
    ms_zoning="RL",
    kitchen_qual="TA",
    central_air="Y"
)

In [49]:
print(user_input.shape)

(1, 283)


In [50]:
prediction_log = gb_model.predict(user_input)

prediction = np.expm1(prediction_log)

print(f"Predicted House Price: ${prediction[0]:,.2f}")

Predicted House Price: $170,165.82


In [51]:
print(user_input.T[user_input.T[0] != 0])

                         0
LotArea               8450
OverallQual              7
OverallCond              5
YearBuilt             2003
YearRemodAdd          2003
TotalBsmtSF            856
1stFlrSF               856
2ndFlrSF               854
GrLivArea             1710
FullBath                 2
HalfBath                 1
BedroomAbvGr             3
TotRmsAbvGrd             7
Fireplaces               1
GarageYrBlt           2003
GarageCars               2
GarageArea             548
MSZoning_RL              1
Neighborhood_CollgCr     1
CentralAir_Y             1
KitchenQual_TA           1
